<a href="https://colab.research.google.com/github/jsmak/dl-framework/blob/main/04_gradient_descent/02_gradientdecent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 경사하강법(Gradient Descent)

섭씨(°C)를 화씨(°F)로 바꾸는 간단한 모델을 PyTorch로 학습시켜 봅니다.
4강에서 배운 **순전파 → 손실 계산 → 역전파 → 경사하강법** 과정이 코드에서 어떻게 동작하는지 확인합니다.

정답 공식은 **F = 1.8 × C + 32** 입니다.
모델에게는 이 공식을 알려 주지 않고, 데이터만 보고 가중치 w(1.8)와 편향 b(32)를 스스로 찾게 합니다.

## 1. 라이브러리 임포트

PyTorch(torch)와 신경망 모듈(nn), 시각화를 위한 matplotlib을 불러옵니다.
매번 같은 결과가 나오도록 난수 시드를 고정합니다.

In [ ]:
import torch
from torch import nn
import matplotlib.pyplot as plt

# 실행할 때마다 같은 초깃값이 나오도록 시드 고정
torch.manual_seed(42)

print("PyTorch 버전:", torch.__version__)

## 2. 데이터 준비

입력 X는 섭씨 온도, 정답 y는 화씨 온도입니다.
데이터는 (10°C → 50°F), (37.78°C → 100°F) 두 개뿐입니다.

텐서의 크기는 (2, 1)입니다. 행은 데이터의 개수, 열은 입력값의 개수입니다.

In [ ]:
# 입력 X: 섭씨 온도
X = torch.tensor([[10.0], [37.78]])

# 정답 y: 화씨 온도
y = torch.tensor([[50.0], [100.0]])

print(X.shape, y.shape)

## 3. 모델, 손실 함수, 옵티마이저 정의

| 코드 | 의미 | 4강 내용 |
|:---|:---|:---|
| `nn.Linear(1, 1)` | 입력 1개, 출력 1개인 모델 **ŷ = w × x + b** | 학습할 파라미터 w, b |
| `nn.MSELoss()` | 평균제곱오차 | 손실함수 L = (ŷ − y)²의 평균 |
| `optim.SGD(lr=0.001)` | 확률적 경사하강법 | 학습률 0.001 |

w와 b는 처음에 무작위 값으로 정해집니다.

In [ ]:
# 모델: y = w * x + b
model = nn.Linear(1, 1)

# 손실 함수: 평균제곱오차(MSE)
loss_fn = nn.MSELoss()

# 옵티마이저: 경사하강법(SGD), 학습률 0.001
optimizer = torch.optim.SGD(model.parameters(), lr=0.001)

# 학습 전 초깃값 저장
init_w = model.weight.item()
init_b = model.bias.item()
print(f"초깃값  w: {init_w:.4f}, b: {init_b:.4f}")

## 4. 학습: 4단계 반복

데이터 전체를 한 번 사용해 아래 과정을 한 바퀴 도는 것이 **1 에폭(Epoch)** 입니다.
여기서는 10,000 에폭을 반복합니다.

0. `optimizer.zero_grad()` : 이전에 계산한 기울기를 0으로 초기화 (기울기는 누적되기 때문)
1. `model(X)` : **순전파**, 현재 w, b로 예측값 계산
2. `loss_fn(outputs, y)` : **손실 계산**, 예측과 정답의 차이
3. `loss.backward()` : **역전파**, w와 b의 기울기 계산
4. `optimizer.step()` : **경사하강법**, W<sub>new</sub> = W<sub>old</sub> − 학습률 × 기울기

In [ ]:
epochs = 10000
snapshot_epochs = [1, 2, 3, 10, 500, 1000, 2000, 5000, 10000]

losses = []                          # 에폭별 손실 기록
snapshots = [(0, init_w, init_b)]    # 시각화용 (에폭, w, b) 기록

for epoch in range(1, epochs + 1):
    # 0. 기울기 초기화
    optimizer.zero_grad()

    # 1. 순전파: 예측값 계산
    outputs = model(X)

    # 2. 손실 계산
    loss = loss_fn(outputs, y)

    # 3. 역전파: 기울기 계산
    loss.backward()

    # 4. 경사하강법: w, b 업데이트
    optimizer.step()

    losses.append(loss.item())

    # 정해 둔 에폭마다 w, b 기록
    if epoch in snapshot_epochs:
        w = model.weight.item()
        b = model.bias.item()
        snapshots.append((epoch, w, b))
        print(f"Epoch {epoch:5d} | loss: {loss.item():10.4f} | w: {w:.4f}, b: {b:.4f}")

## 5. 학습 결과 확인

학습 전과 후의 식을 실제 공식과 비교합니다.

In [ ]:
final_w = model.weight.item()
final_b = model.bias.item()

print(f"실제 공식   F = 1.8000 * C + 32.0000")
print(f"학습 전     F = {init_w:.4f} * C + {init_b:.4f}")
print(f"학습 후     F = {final_w:.4f} * C + {final_b:.4f}")

## 6. 학습 과정 시각화

에폭에 따라 모델의 예측선(주황 점선)이 실제 공식(검은 실선)에 가까워지는 모습과, 손실이 줄어드는 모습을 그래프로 확인합니다.

In [ ]:
# 시각화용 섭씨 0~100도
celsius = torch.linspace(0, 100, 51)
fahrenheit_true = celsius * 1.8 + 32

fig, axes = plt.subplots(2, 5, figsize=(16, 6), sharex=True, sharey=True)
for ax, (epoch, w, b) in zip(axes.flatten(), snapshots):
    ax.plot(celsius, fahrenheit_true, color="black", linewidth=2, label="True")
    ax.plot(celsius, w * celsius + b, "--", color="orange", label="Pred")
    ax.scatter(X, y, color="red", zorder=3)
    ax.set_title(f"Epoch {epoch}")
    ax.grid(True)
axes[0, 0].legend()
fig.supxlabel("Celsius (°C)")
fig.supylabel("Fahrenheit (°F)")
plt.tight_layout()
plt.show()

In [ ]:
# 에폭별 손실 변화 (세로축은 로그 스케일)
plt.figure(figsize=(8, 4))
plt.plot(range(1, epochs + 1), losses)
plt.yscale("log")
plt.xlabel("Epoch")
plt.ylabel("Loss (MSE)")
plt.title("Loss")
plt.grid(True)
plt.show()